In [1]:
!pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 22.7 MB/s eta 0:00:00a 0:00:01


In [6]:
yaml_content = """
path: /kaggle/working/datasets/processed
train: images/train
val: images/train  # Point to your validation folder if you created one

names:
  0: Debris
  1: Marine_Life
"""

with open('/kaggle/working/data.yaml', 'w') as f:
    f.write(yaml_content)

print("data.yaml created successfully!")

data.yaml created successfully!


In [3]:
import os
import cv2
import glob
import numpy as np
import base64
import json
import zlib
from pathlib import Path

# --- 1. CORRECTED SETUP PATHS ---
# Using the exact paths we just found!
RAW_IMAGES = Path('/kaggle/input/datasets/mexwell/trashcan-1-0/dataset/original_data/images')
RAW_ANNOTATIONS = Path('/kaggle/input/datasets/mexwell/trashcan-1-0/dataset/original_data/annotations')

OUTPUT_BASE = Path('/kaggle/working/datasets/processed')
OUT_IMAGES = OUTPUT_BASE / 'images' / 'train'
OUT_LABELS = OUTPUT_BASE / 'labels' / 'train'

OUT_IMAGES.mkdir(parents=True, exist_ok=True)
OUT_LABELS.mkdir(parents=True, exist_ok=True)

CLASS_MAP = {"trash": 0, "debris": 0, "marine_life": 1, "fish": 1}

# --- 2. JSON TO POLYGON LOGIC ---
def decode_bitmap_data(b64_data: str) -> np.ndarray:
    raw = base64.b64decode(b64_data)
    try: raw = zlib.decompress(raw)
    except zlib.error: pass
    buf = np.frombuffer(raw, dtype=np.uint8)
    img = cv2.imdecode(buf, cv2.IMREAD_GRAYSCALE)
    if img is None: return None
    _, mask = cv2.threshold(img, 127, 255, cv2.THRESH_BINARY)
    return mask

def mask_to_polygon(mask: np.ndarray, origin_x: int, origin_y: int, img_w: int, img_h: int):
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    polygons = []
    for c in contours:
        if cv2.contourArea(c) < 9: continue
        pts = c.reshape(-1, 2).astype(np.float32)
        pts[:, 0] += origin_x
        pts[:, 1] += origin_y
        pts[:, 0] /= img_w
        pts[:, 1] /= img_h
        pts = np.clip(pts, 0, 1)
        if len(pts) >= 3:
            polygons.append(pts.flatten().tolist())
    return polygons

def process_annotation(json_path: Path, img_w: int, img_h: int):
    with open(json_path, encoding="utf-8") as f: data = json.load(f)
    results = []
    for obj in data.get("objects", []):
        if obj.get("geometryType") != "bitmap": continue
        bmp = obj.get("bitmap")
        if not bmp: continue
        origin = bmp.get("origin", [0, 0])
        ox, oy = int(origin[0]), int(origin[1])
        mask = decode_bitmap_data(bmp.get("data", ""))
        if mask is None: continue
        polys = mask_to_polygon(mask, ox, oy, img_w, img_h)
        key = (obj.get("classTitle", "") or "").strip().lower().replace(" ", "_")
        cid = CLASS_MAP.get(key, 0) # Default to 0 (Debris)
        for poly in polys:
            results.append((cid, poly))
    return results

# --- 3. MAIN PROCESSING LOOP ---
print("Starting image processing and label conversion...")

# We are processing 500 images for a quick test run. 
# Once this works, you can remove "[:500]" to process all 7,200 images!
image_files = list(RAW_IMAGES.glob('*.jpg')) 

success_count = 0

for img_path in image_files:
    # Find matching JSON
    json_path = RAW_ANNOTATIONS / (img_path.name + ".json")
    if not json_path.is_file():
        json_path = RAW_ANNOTATIONS / (img_path.stem + ".json")
    
    if not json_path.is_file():
        continue # Skip if no annotation exists
        
    # Read Image
    img = cv2.imread(str(img_path))
    if img is None: continue
    h, w = img.shape[:2]
    
    # Process Labels
    polygons = process_annotation(json_path, w, h)
    if not polygons:
        continue # Skip if no valid polygons were found
        
    # Process Image (Masking & Canny)
    mask = np.zeros((h, w), dtype=np.uint8)
    cv2.rectangle(mask, (w-150, h-50), (w, h), 255, -1) 
    inpainted = cv2.inpaint(img, mask, 3, cv2.INPAINT_TELEA)
    edges = cv2.Canny(inpainted, 100, 200)
    edges_colored = cv2.cvtColor(edges, cv2.COLOR_GRAY2BGR)
    final_img = cv2.addWeighted(inpainted, 0.8, edges_colored, 0.2, 0)
    
    # Save Image
    cv2.imwrite(str(OUT_IMAGES / img_path.name), final_img)
    
    # Save Labels
    with open(OUT_LABELS / (img_path.stem + ".txt"), "w") as f:
        for cid, poly in polygons:
            poly_str = " ".join(f"{x:.6f}" for x in poly)
            f.write(f"{cid} {poly_str}\n")
            
    success_count += 1

print(f"DONE! Successfully generated {success_count} paired images and labels.")

Starting image processing and label conversion...


KeyboardInterrupt: 

In [13]:
import yaml

data_config = {
    'train': '/kaggle/working/datasets/processed/images/train',
    'val': '/kaggle/working/datasets/processed/images/train', # Using train for val since we didn't split them
    'nc': 2,
    'names': ['Debris', 'Marine Life']
}

with open('/kaggle/working/data.yaml', 'w') as f:
    yaml.dump(data_config, f)
    
print("Perfect data.yaml created!")

Perfect data.yaml created!


In [14]:
from ultralytics import YOLO

model = YOLO("yolov8n-seg.pt")

results = model.train(
    data="/kaggle/working/data.yaml",
    epochs=50,
    imgsz=640,
    batch=8,       
    device=0,      
    project="/kaggle/working/runs",
    name="marine_debris_run_FULL" # New folder for your final masterpiece
)

Ultralytics 8.4.33 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n-seg.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=marine_debris_run_FULL, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100,

In [2]:
yaml_content = """
path: /kaggle/working/datasets/clahe_processed
train: images/train
val: images/train  

names:
  0: Debris
  1: Marine_Life
"""
with open('/kaggle/working/data_clahe.yaml', 'w') as f:
    f.write(yaml_content)

print("data_clahe.yaml successfully created!")

data_clahe.yaml successfully created!


In [3]:
import os
import cv2
import glob
import numpy as np
import base64
import json
import zlib
from pathlib import Path

# --- 1. SETUP PATHS FOR CLAHE ---
RAW_IMAGES = Path('/kaggle/input/datasets/mexwell/trashcan-1-0/dataset/original_data/images')
RAW_ANNOTATIONS = Path('/kaggle/input/datasets/mexwell/trashcan-1-0/dataset/original_data/annotations')

# NEW Output folder so we don't overwrite the old one!
OUTPUT_BASE = Path('/kaggle/working/datasets/clahe_processed')
OUT_IMAGES = OUTPUT_BASE / 'images' / 'train'
OUT_LABELS = OUTPUT_BASE / 'labels' / 'train'

OUT_IMAGES.mkdir(parents=True, exist_ok=True)
OUT_LABELS.mkdir(parents=True, exist_ok=True)

CLASS_MAP = {"trash": 0, "debris": 0, "marine_life": 1, "fish": 1}

# --- 2. JSON TO POLYGON LOGIC (Unchanged) ---
def decode_bitmap_data(b64_data: str) -> np.ndarray:
    raw = base64.b64decode(b64_data)
    try: raw = zlib.decompress(raw)
    except zlib.error: pass
    buf = np.frombuffer(raw, dtype=np.uint8)
    img = cv2.imdecode(buf, cv2.IMREAD_GRAYSCALE)
    if img is None: return None
    _, mask = cv2.threshold(img, 127, 255, cv2.THRESH_BINARY)
    return mask

def mask_to_polygon(mask: np.ndarray, origin_x: int, origin_y: int, img_w: int, img_h: int):
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    polygons = []
    for c in contours:
        if cv2.contourArea(c) < 9: continue
        pts = c.reshape(-1, 2).astype(np.float32)
        pts[:, 0] += origin_x
        pts[:, 1] += origin_y
        pts[:, 0] /= img_w
        pts[:, 1] /= img_h
        pts = np.clip(pts, 0, 1)
        if len(pts) >= 3:
            polygons.append(pts.flatten().tolist())
    return polygons

def process_annotation(json_path: Path, img_w: int, img_h: int):
    with open(json_path, encoding="utf-8") as f: data = json.load(f)
    results = []
    for obj in data.get("objects", []):
        if obj.get("geometryType") != "bitmap": continue
        bmp = obj.get("bitmap")
        if not bmp: continue
        origin = bmp.get("origin", [0, 0])
        ox, oy = int(origin[0]), int(origin[1])
        mask = decode_bitmap_data(bmp.get("data", ""))
        if mask is None: continue
        polys = mask_to_polygon(mask, ox, oy, img_w, img_h)
        key = (obj.get("classTitle", "") or "").strip().lower().replace(" ", "_")
        cid = CLASS_MAP.get(key, 0)
        for poly in polys:
            results.append((cid, poly))
    return results

# --- 3. MAIN PROCESSING LOOP (WITH CLAHE!) ---
print("Starting CLAHE enhancement and label conversion...")

image_files = list(RAW_IMAGES.glob('*.jpg')) 
success_count = 0

for img_path in image_files:
    json_path = RAW_ANNOTATIONS / (img_path.name + ".json")
    if not json_path.is_file():
        json_path = RAW_ANNOTATIONS / (img_path.stem + ".json")
    if not json_path.is_file(): continue 
        
    img = cv2.imread(str(img_path))
    if img is None: continue
    h, w = img.shape[:2]
    
    polygons = process_annotation(json_path, w, h)
    if not polygons: continue 
        
    # 1. Mask out the dataset watermark
    mask = np.zeros((h, w), dtype=np.uint8)
    cv2.rectangle(mask, (w-150, h-50), (w, h), 255, -1) 
    inpainted = cv2.inpaint(img, mask, 3, cv2.INPAINT_TELEA)
    
    # 2. APPLY CLAHE (The Science Experiment!)
    # Convert to LAB color space
    lab = cv2.cvtColor(inpainted, cv2.COLOR_BGR2LAB)
    l_channel, a_channel, b_channel = cv2.split(lab)
    
    # Apply CLAHE to the L (Lightness) channel
    clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8,8))
    cl = clahe.apply(l_channel)
    
    # Merge back to BGR image
    merged = cv2.merge((cl, a_channel, b_channel))
    clahe_img = cv2.cvtColor(merged, cv2.COLOR_LAB2BGR)
    
    # Save Image
    cv2.imwrite(str(OUT_IMAGES / img_path.name), clahe_img)
    
    # Save Labels
    with open(OUT_LABELS / (img_path.stem + ".txt"), "w") as f:
        for cid, poly in polygons:
            poly_str = " ".join(f"{x:.6f}" for x in poly)
            f.write(f"{cid} {poly_str}\n")
            
    success_count += 1

print(f"DONE! Successfully generated {success_count} CLAHE-enhanced paired images.")

Starting CLAHE enhancement and label conversion...
DONE! Successfully generated 7212 CLAHE-enhanced paired images.


In [4]:
from ultralytics import YOLO

# 1. Load the Small version of YOLOv8
model_v8s = YOLO('yolov8s-seg.pt') 

print("🚀 Starting YOLOv8-Small Benchmarking...")

# 2. Train on your CLAHE dataset
results_v8s = model_v8s.train(
    data='/kaggle/working/data_clahe.yaml', 
    epochs=50, 
    imgsz=640, 
    batch=16, # If Kaggle gives an 'Out of Memory' error, change this to 8
    name="v8_small_clahe_run"
)

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
🚀 Starting YOLOv8-Small Benchmarking...
Ultralytics 8.4.38 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/data_clahe.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=Fa

In [5]:
import shutil
shutil.make_archive('Final_SOTA_Results', 'zip', '/kaggle/working/runs/segment/v8_small_clahe_run')
print("Finished! Download 'Final_SOTA_Results.zip' from the right-hand panel.")

Finished! Download 'Final_SOTA_Results.zip' from the right-hand panel.


In [4]:
import yaml

data_config = {
    'train': '/kaggle/working/datasets/clahe_processed/images/train',
    'val': '/kaggle/working/datasets/clahe_processed/images/train', # Using train for val
    'nc': 2,
    'names': ['Debris', 'Marine_Life']
}

with open('/kaggle/working/data_clahe.yaml', 'w') as f:
    yaml.dump(data_config, f)
    
print("Perfect data_clahe.yaml created!")

Perfect data_clahe.yaml created!


In [10]:
from ultralytics import YOLO

# Load a fresh, untrained YOLO model
model = YOLO('yolov8n-seg.pt')

print("Starting CLAHE Training Run...")

# Train using the new CLAHE dataset map!
results = model.train(
    data='/kaggle/working/data_clahe.yaml', 
    epochs=50, 
    imgsz=640, 
    batch=16,
    name="clahe_run" # Names the output folder
)

print("Done! Go download your new CLAHE results!")

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Starting CLAHE Training Run...
Ultralytics 8.4.33 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/data_clahe.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv

In [5]:
from ultralytics import YOLO

# Load a fresh, completely untrained model so it's a fair test
model = YOLO('yolov8n-seg.pt')

print("Starting Phase 3: Augmented CLAHE Training Run...")

results = model.train(
    data='/kaggle/working/data_clahe.yaml', 
    epochs=50, 
    imgsz=640, 
    batch=16,
    name="clahe_augmented_run", # NEW FOLDER NAME!
    
    # --- RESEARCH-GRADE AUGMENTATIONS ---
    
    # 1. Spatial Transformations
    degrees=45.0,    # Rotate images up to 45 degrees (trash floats at all angles)
    fliplr=0.5,      # 50% chance to flip left-right
    flipud=0.5,      # 50% chance to flip upside down (no gravity underwater)
    
    # 2. Complex Blending
    mosaic=1.0,      # 100% chance to stitch 4 images into 1 (forces it to find small objects)
    mixup=0.15,      # 15% chance to blend two images together (simulates murky transparency)
    
    # 3. Scale & Perspective
    scale=0.5,       # Zoom in/out by 50%
    perspective=0.0  # Keep at 0 (perspective warping can sometimes break bounding boxes)
)

print("Done! Go download your Phase 3 Augmented results!")

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Starting Phase 3: Augmented CLAHE Training Run...
Ultralytics 8.4.34 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/data_clahe.yaml, degrees=45.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.5, format=torchscript, fraction=1.0, freeze=None, half=Fal